# Import Libraries

In [ ]:
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix

# Create Meta Data

**carID:** An atribute that contains an identifier for each car.

**Brand:** The cars main brand (e.g., Ford, Toyota).

**model:** The car model.

**year:** The year of registration of the car.

**mileage:** The total reported distance travelled by the car (in miles).

**tax:** The amount of road tax (in £) that, in 2020, was applicable to the car in question.

**fuelType:** Type of Fuel used by car (Diesel, Petrol, Hybrid, Electric).

**mpg:** Average Miles per Gallon.

**engineSize:** Size of Engine in liters (Cubic Decimeters).

**paintQuality%** The mechanic’s assessment of the cars’ overall paint quality and hull integrity (filled by the mechanic during evaluation). 

**previousOwner:** Number of previous registered owners of the vehicle.

**hasDamage:** Boolean marker filled by the seller at the time of registration stating whether the car is damaged or not.

**price:** The car's price when purchased by Cars 4 You (in £).

# Import Dataset

In [ ]:
sample = pd.read_csv('sample_submission.csv')
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

# EDA (Exploratory Data Analysis)

## Define new index for our datasets

In [ ]:
sample.set_index('carID', inplace = True)
train.set_index('carID', inplace = True)
test.set_index('carID', inplace = True)

## Sample

In [ ]:
sample.shape

In [ ]:
sample.head()

In [ ]:
sample.tail()

In [ ]:
#From this result we can see that we don't have missing values on this data set
sample.info()

In [ ]:
sample.describe()

## Train

In [ ]:
train.shape

In [ ]:
train.head()

In [ ]:
train.tail()

From the visualization of the head and tail of the data base we can already understand that some errors exist:

    - Missing values
    - Values that should be integers as floats (2020.0)
We will further analyse this using describe and info.

It's also possible to see that some strings have the same information written in different forms (Diesel, iesel; Mercedes, mercedes).
To solve this problem we will uniformize all the values in data preparation

In [ ]:
train.info()

From info we can see that:

    - year as a float...
    - previousOwners also as floats but they should be integers and booleans respectively
    - Missing values in all features except the price 

What will we do?

    Analyse with describe to have a different view

In [ ]:
train.describe()

From the numeric describe we can see that we have some weird values:

    1. negative mileage, tax, mpg, engineSize, previousOwners in the minimum value
    2. hasDamage is a boolean but we can see that instead of 0 and 1 we only have 0 and Nones*
    3. previousOwner has a float? Should we round it?

*check in the hasDamage column

What will we do:

    1. Count the number of negative values and decide if we should drop or change them.
    2. Replace the nones by 1's. (data-preparation)
    3. Count the number of float values and decide to drop or round them.

In [ ]:
train.describe(include='object')

From the categorical describe we can confirm that this columns also have missing values


In [ ]:
train.unique()

### Check for duplicates

In [ ]:
train.duplicated().sum()
#We conclude that there aren't any duplicates on the whole table

Besides not having duplicates it's important to examine again the duplicates without the carID, to prevent redundancy.

In [ ]:
#First we create a copy of our dataFrame 
train_copy= train.copy()
train_copy.head()

In [ ]:
#We drop carID
train_without_ID = train_copy.drop('carID', axis=1)
train_without_ID.head() #to check that everything is okay

In [ ]:
train_without_ID.duplicated().sum() 
#count the total of duplicates of our new DataFrame

After this analysis we found inly 4 duplicates, we decide to drop the duplicate lines. 

### GroupBy

In [ ]:
train.groupby('hasDamage')['price'].mean()

From the code before we can see that the price mean for the cars that have no damage is 16883.212509. 
It would also make sense to analyse the mean price of the cars with damage but the problem is that assume that we have cars with damage, because the column with the feature hasDamge has no 1's.
this way it's impossible to check the mean price of the cars with damage.

In [ ]:
train.groupby('mileage')['price'].mean()
#Here we can check a decrease of the price with the increase of the mileages
#But we still have the problem of the negatives mileages

In [ ]:
train.groupby('year')['price'].mean()
#It's possible to verify that the price varies depending ont he year of manufacture of the cars. 
#This is, the older the car, he cheaper it is

In [ ]:
!!!!! O CÓDIGO A SEGUIR FORAM RACIOCÍNEOS QUE FIZ QUE PODIAM DAR RESULTADOS GIROS MAS QUE NÃO SEI SE VALE A PENA POR  !!!!

train.groupby('previousOwners')['price'].mean()
#aqui podiamos relacionar o aumento do nº de donos com a diminuição do preço mas acho que os resultados não mostram nada muito
#interessante, não sei se vale a pena por

train.groupby('Brand')['price'].mean()
#pode ser interessante ver depois de ter corrigido o problema dos erros na escrita de cada marca. 
# é claro pelo resultado que por exemplo os yundais sao mais baratos que maior parte das outras marcas

train.groupby('paintQuality%')['price'].mean()
#acho que o resultado não mostra nada que tenha uma boa conclusao

In [ ]:
!!!!! O CÓDIGO A SEGUIR FORAM RACIOCÍNEOS QUE FIZ QUE PODIAM DAR RESULTADOS GIROS MAS QUE NÃO SEI SE VALE A PENA POR  !!!!

train.groupby('year')['paintQuality%'].mean()
#poderiamos ver que quanto mais velho o carro pior a qualidade de pintura, mas acho que isso não é muito explicito pelos resultados

train.groupby('previousOwners')['paintQuality%'].mean()
#os resultados tb nao permitem concluir nada

train.groupby('hasDamage')['paintQuality%'].mean()
#acho que aqui era giro comparar a qualidade de pintura com os carros que têm e não têm estragos mas como só temos missing values nos sitios dos 1's nao conseguimos concluir nada disso



### Define the independent variables as X and the dependent as Y

In [ ]:
X = train.drop('price', axis = 1)
y = train['price']

TO DO:
- corrigir types
- remover outliers graves, que são erros e visualizar boxplots
- Feature engineering: Criar features que não envolvem cálculos c/ média, mediana, …
Fazer tudo isto para o training and test set

### Split the dataset into train and validation

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(X,y, test_size = 0.3, 
                                                  random_state = 0, 
                                                  #stratify = y- não pus esta parte como no notebook pq estava a dar erro e acho que é pq a nossa variavel y aqui não é um boolean mas sim um float
                                                  shuffle = True)

TO DO:

- Feature engineering: Criar features que envolvem cálculos c/ média, mediana, … (fazer para os 3 sets que temos)
- Preencher missing values nos 3 datasets, acho que dá para usar uma função ‘transform’ (justificar para depois por no report)  
!!!!!ATENÇÃO: os valores de média, moda,… a usar neste últimos dois passos são todos retirados apenas do training set e não do validation set

### Fill missing values

As we saw there are missing values in a couple of variables so we will fill the categories missing values with 'Unknow' and the numericals with the average.

But before do that we'll split our columns in metric and non_metric features.

In [ ]:
for column in ['Brand', 'model', 'transmission', 'fuelType']:
    X_train[column] = X_train[column].fillna('Unknown')
    X_val[column] = X_val[column].fillna('Unknown')

In [ ]:
for column in X_train.columns:
    if pd.api.types.is_numeric_dtype(X_train[column]):
        
        #store mean of training data in a variable - in a real application, you may need to store these values for future usages on e.g. test data 
        mean_to_fill = X_train[column].mean()
        
        #fill on X_train
        X_train[column].fillna(mean_to_fill, inplace=True)
        #Fill on X_val
        X_val[column].fillna(mean_to_fill, inplace=True)

## Test

In [ ]:
test.shape
#Here we can see that the test shape is equal to the sample shape.

In [ ]:
test.head()

In [ ]:
test.tail()

Errors to account for (reviewed in here and in excel filters):

- NaN in all columns
- Automatic for example sometimes is in CAPS sometimes in lower case 
- Year different than integer
- Missing letters like Manual as anual in transmission column etc (also in fuelType)
- PaintQuality% greater than 100, it's not possible if it's a percentage. Some values are big floats and most of them are integers

From the code before we can see that:
- year, hasDamage, previousOwners as a float
- theres NaN values in all columns

Errors to account for (reviewed in here and in excel filters):

- Missing letters like Manual as anual in transmission column etc (also in fuelType)
- Floats on mpg, previousOwners column with diferent sizes
- NaN values in tax column, previousOwners
- hasDamage, it only contains 0's and blanks/None values, are the blanks supose to be 1's?
- Missing values in all columns
- PreviousOwner and tax: negative values are impossible

# Data Preprocessing

What should we do with the negative values? First of all we'll analyse them.

In [ ]:
mileage_train_negatives = train['mileage']<0
tax_train_negatives = train['tax']<0
mpg_train_negatives = train['mpg']<0
engineSize_train_negatives = train['engineSize']<0
previousOwners_train_negatives = train['previousOwners']<0


#here we used chatGPT to help us to construct the following DataFrame
negatives_summary = pd.DataFrame({
    'mileage_negatives': mileage_train_negatives.value_counts(),
    'tax_negatives': tax_train_negatives.value_counts(),
    'mpg_negatives': mpg_train_negatives.value_counts(),
    'engineSize_negatives': engineSize_train_negatives.value_counts(),
    'previousOwners_negatives': previousOwners_train_negatives.value_counts()
})

negatives_summary

In [ ]:
#Check the percentage of negative mileage values
data_len = len(train['mileage'])

#Here we used chatGPT to help us constructing the final negatives_table
negatives_percent = {
    'mileage_negatives (%)': mileage_train_negatives.sum() / data_len * 100,
    'tax_negatives (%)': tax_train_negatives.sum() / data_len * 100,
    'mpg_negatives (%)': mpg_train_negatives.sum() / data_len * 100,
    'engineSize_negatives (%)': engineSize_train_negatives.sum() / data_len * 100,
    'previousOwners_negatives (%)': previousOwners_train_negatives.sum() / data_len * 100
}

# Converter para DataFrame (tabela)
negatives_table = pd.DataFrame(negatives_percent, index=['Percentage of Negatives']).round(3)

negatives_table

From the negatives_table we obtain very low percentages for each variable, lower than 1%. So it's possible to conclude that this values are a small part of our data and that if we drop them we don't loose veracity.

We can also see that the total percentage of negative values is lower then 5%.

In [ ]:
# The goal is to count the values that are floats, this is, the values that have decimals different than 0
# To do this we create a variable with only the values that have decimals cases different than 0, that is, 
#the values for which the remainder of the division by 1 is not 0
has_decimals = train['previousOwners'] % 1 != 0
print(has_decimals.sum())
#at the end we sum all those values to find how many "real" floats exist


has_decimals.sum() / data_len * 100 

In [ ]:
brand_same_size = train['Brand'].str.lower()
brand_same_size.value_counts().to_frame()

In [ ]:
mapping_brands = {
    'ord' : 'ford',
    'for' : 'ford',
    'ercedes' : 'mercedes',
    'mercede' : 'mercedes',
    'w' : 'vw',
    'v' : 'vw',
    'ope' : 'opel',
    'pel' : 'opel',
    'mw' : 'bmw',
    'aud' : 'audi',
    'udi' : 'audi',
    'bm' : 'bmw',
    'oyota' : 'toyota',
    'koda' : 'skoda',
    'skod' : 'skoda',
    'toyot' : 'toyota',
    'yundai' : 'hyundai',
    'hyunda' : 'hyundai',
    'ercede' : 'mercedes',
    'or' : 'ford',
    'pe' : 'opel',
    'yunda' : 'hyundai',
    'ud' : 'audi',
    'kod' : 'skoda'
}
#train['Brand'] = train['Brand].replace(mapping_brands)

podemos usar esta maneira ou uma biblioteca chamada fuzzy matching mas não sei se vai ser permitido

hasDamage column - replace None values by 1

In [ ]:
train['hasDamage']=train['hasDamage'].fillna(1) #do the same for test set?

In [ ]:
#validate the changes, isto podemos apagar depois
train['hasDamage'].value_counts() 

previousOwner column - round to transform float into int

In [ ]:
train['previousOwners'] = train['previousOwners'].round(0) #do the same for test set?

In [ ]:
train['previousOwners'].value_counts() #negative values remove or stay? isto podemos apagar depois